# Marketing AI Agent

**Prototype prepared using Google's Gemini API (free tier) and Google Colab**

---

## Overview

This notebook is a working prototype of an AI marketing assistant. Given a client brief, it:

- Writes marketing content (social posts, ad copy, captions, email subject lines) tailored to the brief
- Automatically checks that each piece of content fits the target platform's character limit, and revises it if not
- Compiles finished content into a content calendar, exportable as a CSV file

This is an **agentic** system: rather than only generating text, it uses small tools to check and record its own work, the way a human assistant would consult a style guide or maintain a shared tracker while completing a task.

**Cost:** This version runs on the Gemini API free tier. No billing account or credit card is required. Free usage is limited to a daily request quota (generous for individual or small-team use); Google's current limits are visible at https://ai.google.dev/gemini-api/docs/rate-limits.

**Data note:** On the free tier, Google may use submitted prompts to improve its products. Avoid entering confidential client data until this is upgraded to a paid tier with a no-training guarantee, if that matters for your clients.

---

## How to run this notebook

Run each cell in order, from top to bottom, using the play button to the left of each cell. Instructions are provided above each step.

## Step 1 — Install dependencies

Installs the Google Gemini SDK and pandas (used to build the content calendar table).

In [ ]:
!pip install -q google-genai pandas

## Step 2 — Connect to the Gemini API

A free API key is required. To get one:

1. Go to https://aistudio.google.com/apikey
2. Sign in with a Google account
3. Click **Create API key** and copy it

No payment details are required for this step. Run the cell below and paste the key when prompted; it will not be displayed or saved in this file.

In [ ]:
import getpass
from google import genai
from google.genai import types

api_key = getpass.getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=api_key)

MODEL = "gemini-2.5-flash"  # Free-tier model; strong quality-to-cost balance

print("Connected.")

Paste your Gemini API key: ··········
Connected.


## Step 3 — Define the agent's tools

Two functions give the agent capabilities beyond plain text generation:

- `check_platform_limits` — verifies content length against common platform limits
- `save_content` — records a finished piece of content in the content calendar

The model decides on its own when to call these while completing a request. No manual code changes are needed to run this cell.

In [ ]:
import pandas as pd

content_calendar = []

PLATFORM_LIMITS = {
    "twitter": 280,
    "x": 280,
    "instagram_caption": 2200,
    "facebook": 63206,
    "linkedin": 3000,
    "google_ad_headline": 30,
    "email_subject_line": 60,
}

def check_platform_limits(platform: str, text: str) -> dict:
    """Checks whether marketing content fits the character limit of a given platform.

    Args:
        platform: Platform name (twitter, instagram_caption, facebook, linkedin,
            google_ad_headline, or email_subject_line).
        text: The content to check.
    """
    limit = PLATFORM_LIMITS.get(platform.lower().replace(" ", "_"))
    length = len(text)
    if limit is None:
        return {"platform": platform, "length": length, "limit": "unknown", "fits": True}
    return {"platform": platform, "length": length, "limit": limit, "fits": length <= limit}

def save_content(client_name: str, platform: str, content: str, notes: str = "") -> dict:
    """Saves a finished, approved piece of marketing content to the content calendar.

    Args:
        client_name: Name of the client the content is for.
        platform: Target platform for the content.
        content: The finished content text.
        notes: Optional notes, e.g. a suggested posting time.
    """
    content_calendar.append({"client": client_name, "platform": platform, "content": content, "notes": notes})
    return {"status": "saved", "total_items_saved": len(content_calendar)}

print("Tools defined.")

Tools defined.


## Step 4 — Define the agent

This wraps the model call, system instructions, and tools into a single reusable function. Run this cell once; no edits needed.

In [ ]:
SYSTEM_INSTRUCTIONS = (
    "You are a professional marketing assistant working for a marketing agency. "
    "Given a client brief, write high-quality marketing content (social posts, ad copy, "
    "email subject lines, captions, etc). Always use check_platform_limits on any content "
    "written for a specific platform, and revise it if it doesn't fit. Once a piece of "
    "content is finished and confirmed to fit, save it with save_content. Close with a "
    "brief summary of what was produced."
)

def run_marketing_agent(brief: str) -> str:
    """Runs the marketing agent on a client brief and returns its final summary."""
    response = client.models.generate_content(
        model=MODEL,
        contents=brief,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTIONS,
            tools=[check_platform_limits, save_content],
        ),
    )
    print(response.text)
    return response.text

print("Agent ready.")

Agent ready.


## Step 5 — Run a sample brief

The cell below includes a sample client brief to confirm everything works end to end.

In [ ]:
global MODEL
MODEL = "gemini-3.6-flash"

sample_brief = """
Client: GreenSip (an eco-friendly reusable water bottle brand)
Task: We're launching a new insulated bottle in matte green. Write:
1. One Twitter/X post announcing the launch
2. One Instagram caption for the same launch, more descriptive, with relevant hashtags
3. One email subject line to announce it to our newsletter list
Tone: upbeat, eco-conscious, friendly, not overly salesy.
"""

run_marketing_agent(sample_brief)

Here is the marketing content written for **GreenSip's** new Matte Green insulated bottle launch:

---

### 1. Twitter/X Post
> Meet your new daily hydration buddy! 🌿 Say hello to our brand-new insulated bottle in Matte Green. Keeps your drinks icy cold (or cozy warm) while keeping single-use plastic out of our oceans. Tap the link to shop the launch! 💧✨ #GreenSip #StayHydrated

---

### 2. Instagram Caption
> Nature called, and it said your hydration routine needed an upgrade. 🌿✨
> 
> Introducing our newest GreenSip family member: the Insulated Bottle in Matte Green! Designed to keep your favorite drinks ice-cold for 24 hours or piping hot for 12, all wrapped in a sleek, velvety matte finish.
> 
> Whether you're hitting the trail, powering through a workday, or relaxing in the park, this eco-conscious bottle is built to last—helping you ditch single-use plastic in style. 🌍💧
> 
> Ready to make the switch? Tap the link in our bio to shop the launch!
> 
> #GreenSip #MatteGreen #EcoFriend

"Here is the marketing content written for **GreenSip's** new Matte Green insulated bottle launch:\n\n---\n\n### 1. Twitter/X Post\n> Meet your new daily hydration buddy! 🌿 Say hello to our brand-new insulated bottle in Matte Green. Keeps your drinks icy cold (or cozy warm) while keeping single-use plastic out of our oceans. Tap the link to shop the launch! 💧✨ #GreenSip #StayHydrated\n\n---\n\n### 2. Instagram Caption\n> Nature called, and it said your hydration routine needed an upgrade. 🌿✨\n> \n> Introducing our newest GreenSip family member: the Insulated Bottle in Matte Green! Designed to keep your favorite drinks ice-cold for 24 hours or piping hot for 12, all wrapped in a sleek, velvety matte finish.\n> \n> Whether you're hitting the trail, powering through a workday, or relaxing in the park, this eco-conscious bottle is built to last—helping you ditch single-use plastic in style. 🌍💧\n> \n> Ready to make the switch? Tap the link in our bio to shop the launch!\n> \n> #GreenSip #Ma

## Step 6 — Run a real client brief

Replace the text below with an actual client brief, then run the cell. This can be repeated for as many clients or tasks as needed — every run adds to the same content calendar below.

In [ ]:
client_brief = """
Client: [client name]
Task: [what content is needed, and for which platforms]
Tone: [describe the desired tone]
"""

run_marketing_agent(client_brief)

I am ready to help! Please provide the details for the brief by filling in:

- **Client Name:**
- **Task & Target Platforms:** (e.g., Twitter post, Instagram caption, LinkedIn post, Email subject line, Google Ad headline, Facebook post)
- **Desired Tone:** (e.g., professional, playful, urgent, inspiring, educational)
- **Key Message / Product Details:** (Any specific offer, call to action, or details to include)

Once you provide the brief, I will craft the content, verify character limits for each platform, save the finalized pieces to the content calendar, and present a summary for you!


'I am ready to help! Please provide the details for the brief by filling in:\n\n- **Client Name:**\n- **Task & Target Platforms:** (e.g., Twitter post, Instagram caption, LinkedIn post, Email subject line, Google Ad headline, Facebook post)\n- **Desired Tone:** (e.g., professional, playful, urgent, inspiring, educational)\n- **Key Message / Product Details:** (Any specific offer, call to action, or details to include)\n\nOnce you provide the brief, I will craft the content, verify character limits for each platform, save the finalized pieces to the content calendar, and present a summary for you!'

## Step 7 — Review and export the content calendar

In [ ]:
df = pd.DataFrame(content_calendar)
df

,client,platform,content,notes
0,GreenSip,twitter,Meet your new daily hydration buddy! 🌿 Say hel...,Launch announcement post
1,GreenSip,instagram_caption,"Nature called, and it said your hydration rout...",Launch announcement caption with hashtags
2,GreenSip,email_subject_line,"Meet Matte Green 🌿 Our sleekest, coldest bottl...",Newsletter launch announcement subject line


In [ ]:
df.to_csv("content_calendar.csv", index=False)

from google.colab import files
files.download("content_calendar.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>